# Module 6: exercises  (solutions)

Each plotting exercise is a **function that returns the matplotlib `Axes`**, which is the same shape your project's `plots.py` will use.
The checks inspect the Axes, so label your charts!

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

## Exercise 1

**Histogram.** Write `hist_scores(df)` that draws a histogram of `exam_score` with **20 bins** and returns the Axes.
Set a non-empty title and x-label.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

students = pd.read_csv(DATA / "students.csv")
rides = pd.read_csv(DATA / "daily_rides.csv", parse_dates=["date"])

def hist_scores(df):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(df["exam_score"], bins=20, edgecolor="white")
    ax.set_title("Distribution of exam scores")
    ax.set_xlabel("Exam score")
    ax.set_ylabel("Number of students")
    return ax

In [ ]:
ax = hist_scores(students)
assert len(ax.patches) == 20, f"Expected 20 bars (bins=20), found {len(ax.patches)}"
assert ax.get_title() and ax.get_xlabel(), "Add a title and an x-label"
print("✅ Correct!")

## Exercise 2

**Bar chart by weekday.** Write `bar_rides_by_weekday(rides)` that draws the **average daily rides for each weekday**, ordered Monday → Sunday
(hint: `rides["date"].dt.dayofweek` is 0 for Monday). Return the Axes with a y-label.

In [ ]:
def bar_rides_by_weekday(rides):
    fig, ax = plt.subplots(figsize=(6, 4))
    means = rides.groupby(rides["date"].dt.dayofweek)["rides"].mean()
    ax.bar(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], means.values)
    ax.set_ylabel("Average rides per day")
    ax.set_title("Weekends are quieter")
    return ax

In [ ]:
ax = bar_rides_by_weekday(rides)
expected = rides.groupby(rides["date"].dt.dayofweek)["rides"].mean().values
heights = [p.get_height() for p in ax.patches]
assert len(heights) == 7, f"Expected 7 bars, found {len(heights)}"
assert np.allclose(heights, expected), "Bars should show mean rides, Monday first"
assert ax.get_ylabel(), "Add a y-label"
print("✅ Correct!")

## Exercise 3

**Scatter plot.** Write `scatter_study_score(df)` drawing `exam_score` (y) against `study_hours_week` (x), one point per student,
with a title and both axis labels. Return the Axes.

In [ ]:
def scatter_study_score(df):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(df["study_hours_week"], df["exam_score"], alpha=0.6)
    ax.set_xlabel("Study hours per week")
    ax.set_ylabel("Exam score")
    ax.set_title("More study, higher scores")
    return ax

In [ ]:
ax = scatter_study_score(students)
assert len(ax.collections) >= 1 and len(ax.collections[0].get_offsets()) == 200, "One point per student (200)"
assert ax.get_title() and ax.get_xlabel() and ax.get_ylabel(), "Add a title and both axis labels"
print("✅ Correct!")

## Exercise 4

**Correlation by hand and by pandas.** Compute `r_study`: the correlation between `study_hours_week` and `exam_score`, rounded to 2 decimals.
Then find `best_predictor`: the column (other than `exam_score` itself) with the largest **absolute** correlation with `exam_score`.
Hint: `students.corr(numeric_only=True)["exam_score"]` gives all of them; `.drop(...)`, `.abs()` and `.idxmax()` will help.

In [ ]:
r_study = round(students["study_hours_week"].corr(students["exam_score"]), 2)
corr_with_score = students.corr(numeric_only=True)["exam_score"].drop("exam_score").abs()
best_predictor = corr_with_score.idxmax()

In [ ]:
assert r_study == 0.72, f"Got {r_study}"
assert best_predictor == "study_hours_week", f"Got {best_predictor}"
print("✅ Correct!")

## Exercise 5

**Box plot by group.** Write `box_by_major(df)` that draws a box plot of `exam_score` for each `major` (four boxes) and returns the Axes.
Include a title.

In [ ]:
def box_by_major(df):
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.boxplot(data=df, x="major", y="exam_score", ax=ax)
    ax.set_title("Exam score by major")
    return ax

In [ ]:
ax = box_by_major(students)
labels = {t.get_text() for t in ax.get_xticklabels()}
assert labels == {"Arts", "Business", "Engineering", "Science"}, f"x labels: {labels}"
assert ax.get_title(), "Add a title"
print("✅ Correct!")

## Exercise 6

**Fix a misleading chart.** Write `honest_bar(means)` for a Series of two group means. It should draw a bar chart whose y-axis **starts at 0**,
with a y-label. Return the Axes.

In [ ]:
def honest_bar(means):
    fig, ax = plt.subplots(figsize=(4, 3.5))
    ax.bar(means.index, means.values)
    ax.set_ylim(0, 100)
    ax.set_ylabel("Mean exam score")
    return ax

In [ ]:
ax = honest_bar(students.groupby("part_time_job")["exam_score"].mean())
assert len(ax.patches) == 2
assert ax.get_ylim()[0] == 0, "A bar chart's y-axis must start at zero"
assert ax.get_ylabel(), "Add a y-label"
print("✅ Correct!")

## Exercise 7

**Correlation or cause?** Compute `r_temp` = the correlation between `temp_c` and `rides` in `rides` (2 decimals; pandas ignores missing values).
Then choose the best statement and set `answer` to its letter:

* **a)** Warm weather *causes* each individual rider to decide to ride.
* **b)** Rides and temperature move together; weather plausibly drives riding, but the correlation alone doesn't prove how, and other factors
  (weekends, rain, season) could also play a role.
* **c)** The correlation proves temperature causes rides.
* **d)** Since r < 1, there is no relationship.

In [ ]:
r_temp = round(rides["temp_c"].corr(rides["rides"]), 2)
answer = "b"

In [ ]:
assert r_temp == round(rides["temp_c"].corr(rides["rides"]), 2) and r_temp > 0.5
assert answer == "b", "Correlation shows association; be careful with cause and effect"
print(f"✅ Correct! r = {r_temp}")